In [ ]:
# ============================================================
# EVALUACIÓN 2 - Modelos Matemáticos y Simulación
# Detección de daño estructural por grietas (YOLO + Control Estadístico de Procesos)
# Integrantes: ____________________  |  Dataset: Crack-Seg (Ultralytics)
# ============================================================

# PASO 1: Instalar las librerías (Ultralytics = YOLO, Gradio = para subir una imagen)
!pip install -q ultralytics gradio

## FASE I: Dataset
**Dataset:** Crack-Seg (Ultralytics). Fotos de grietas en calles y paredes, con la máscara de la grieta ya dibujada (segmentación).
- 1 sola clase: grieta (crack)
- 4 028 imágenes: 3 717 entrenamiento (train), 200 validación (val) y 111 prueba (test)

**Variables que vamos a extraer de cada grieta:** % de área dañada, longitud, ancho promedio y cantidad de grietas.

In [ ]:
# PASO 2: Leer el archivo .ndjson (trae la información del dataset y la dirección de cada imagen)
import json, requests, random, cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

with open("crack_p1.ndjson") as a, open("crack_p2.ndjson") as b:
    lineas = [l for l in (a.read().splitlines() + b.read().splitlines()) if l.strip()]

cabecera = json.loads(lineas[0])          # primera línea = datos del dataset
imagenes = [json.loads(l) for l in lineas[1:]]   # resto = una línea por imagen

print("Dataset:", cabecera["name"])
print("Descripción:", cabecera["description"])
print("Clases:", cabecera["class_names"])
print("Total de imágenes:", len(imagenes))
print(pd.Series([i["split"] for i in imagenes]).value_counts())

In [ ]:
# PASO 3: Descargar las imágenes y guardar las etiquetas en el formato que pide YOLO
from concurrent.futures import ThreadPoolExecutor

raiz = Path("crack_dataset")
for s in ["train", "val", "test"]:
    (raiz / "images" / s).mkdir(parents=True, exist_ok=True)
    (raiz / "labels" / s).mkdir(parents=True, exist_ok=True)

def descargar(img):
    s = img["split"]
    ruta_img = raiz / "images" / s / img["file"]
    if not ruta_img.exists():
        ruta_img.write_bytes(requests.get(img["url"], timeout=60).content)
    # cada grieta = clase + puntos del contorno
    segmentos = img.get("annotations", {}).get("segments", [])
    texto = "\n".join(str(int(seg[0])) + " " + " ".join(str(v) for v in seg[1:]) for seg in segmentos)
    (raiz / "labels" / s / (Path(img["file"]).stem + ".txt")).write_text(texto)

with ThreadPoolExecutor(16) as ex:
    list(ex.map(descargar, imagenes))

# archivo que le dice a YOLO dónde están los datos
(raiz / "data.yaml").write_text(
    f"path: {raiz.resolve()}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n  0: crack\n")

for s in ["train", "val", "test"]:
    print(s, "->", len(list((raiz / "images" / s).glob("*"))), "imágenes")

In [ ]:
# PASO 4: Ver ejemplos con la grieta marcada en rojo
def dibujar(ruta_img, ruta_lbl):
    img = cv2.cvtColor(cv2.imread(str(ruta_img)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    for linea in Path(ruta_lbl).read_text().splitlines():
        v = list(map(float, linea.split()))[1:]
        pts = (np.array(v).reshape(-1, 2) * [w, h]).astype(np.int32)
        cv2.polylines(img, [pts], True, (255, 0, 0), 2)
    return img

muestras = random.Random(1).sample(list((raiz / "images" / "train").glob("*.jpg")), 6)
fig, ax = plt.subplots(2, 3, figsize=(12, 8))
for a, m in zip(ax.ravel(), muestras):
    a.imshow(dibujar(m, raiz / "labels" / "train" / (m.stem + ".txt")))
    a.axis("off")
plt.suptitle("Grietas marcadas (en rojo) por las etiquetas")
plt.show()

## FASE II: Entrenamiento del modelo YOLO
Usamos **YOLO11 nano de segmentación** (el más liviano), partiendo de pesos preentrenados (*fine-tuning*). Hiperparámetros elegidos por los recursos del Colab (GPU T4 gratuita):
| Parámetro | Valor | Por qué |
|---|---|---|
| epochs | 30 | alcanza para converger y entra en el tiempo |
| imgsz | 416 | es el tamaño original de las imágenes |
| batch | 16 | cabe en la memoria de la GPU |
| optimizador | auto (AdamW, lr = 0.002) | con `optimizer="auto"` YOLO lo elige solo e ignora el `lr0=0.01` que se pasa en el código |

In [ ]:
# PASO 5: Entrenar el modelo (tarda unos 10 a 15 minutos)
from ultralytics import YOLO

modelo = YOLO("yolo11n-seg.pt")     # modelo base ya preentrenado
modelo.train(data=str(raiz / "data.yaml"), epochs=30, imgsz=416, batch=16, lr0=0.01,
             project="runs", name="grietas", exist_ok=True, plots=True)

In [ ]:
# PASO 6: Curvas de entrenamiento y métricas con el mejor modelo (best.pt)
from IPython.display import Image, display
carpeta = Path(modelo.trainer.save_dir)        # carpeta donde YOLO guardó el entrenamiento
display(Image(str(carpeta / "results.png")))     # curvas de pérdida, precisión, recall y mAP

mejor = YOLO(str(carpeta / "weights" / "best.pt"))   # el mejor modelo guardado
m = mejor.val(data=str(raiz / "data.yaml"), split="test", verbose=False)
print("Resultados en las imágenes de PRUEBA (máscara de la grieta):")
print("  Precisión     :", round(float(m.seg.mp), 3))
print("  Recall        :", round(float(m.seg.mr), 3))
print("  mAP@0.5       :", round(float(m.seg.map50), 3))
print("  mAP@0.5:0.95  :", round(float(m.seg.map), 3))

In [ ]:
# PASO 7: Predecir en imágenes de prueba y ver cómo segmenta las grietas
rutas = random.Random(3).sample(list((raiz / "images" / "test").glob("*.jpg")), 6)
fig, ax = plt.subplots(2, 3, figsize=(12, 8))
for a, r in zip(ax.ravel(), rutas):
    res = mejor.predict(str(r), conf=0.25, verbose=False)[0]
    a.imshow(res.plot()[..., ::-1])
    a.axis("off")
plt.suptitle("Predicción del modelo en imágenes de prueba")
plt.show()

## FASE III: Modelo matemático de Control de Calidad
**Idea:** cada imagen es una *pieza inspeccionada*. Con YOLO medimos cuánta grieta tiene y usamos **Control Estadístico de Procesos (SPC)** para decidir si el daño es **BAJO, MEDIO o ALTO**.

Pasos: (1) medir la grieta, (2) definir el proceso base y la **carta de control**, (3) clasificar el daño, (4) calcular la **probabilidad de falla**, (5) simular escenarios (sismo, suelo, construcción), (6) subir una imagen y obtener el diagnóstico.

In [ ]:
# PASO 8: Medir la grieta (variables físicas que extrae el sistema)
from skimage.morphology import skeletonize

def medir_mascara(mascara):
    area = int(mascara.sum())                        # píxeles de grieta
    esqueleto = skeletonize(mascara.astype(bool))    # línea central de la grieta
    longitud = int(esqueleto.sum())                  # largo aproximado en píxeles
    ancho = area / longitud if longitud > 0 else 0.0 # ancho promedio = área / largo
    return area, longitud, ancho

def medir_prediccion(res):
    # mide la grieta que detectó YOLO en una imagen
    h, w = res.orig_shape
    mascara = np.zeros((h, w), np.uint8)
    n = 0
    if res.masks is not None:
        for poly in res.masks.xy:
            if len(poly) >= 3:
                cv2.fillPoly(mascara, [poly.astype(np.int32)], 1)
                n += 1
    area, longitud, ancho = medir_mascara(mascara)
    return {"area_pct": 100 * area / (h * w), "longitud_px": longitud, "ancho_px": round(ancho, 2), "n_grietas": n}

def medir_etiqueta(ruta_lbl, h, w):
    # mide la grieta real (etiqueta dibujada por una persona)
    mascara = np.zeros((h, w), np.uint8)
    n = 0
    for linea in Path(ruta_lbl).read_text().splitlines():
        v = list(map(float, linea.split()))[1:]
        pts = (np.array(v).reshape(-1, 2) * [w, h]).astype(np.int32)
        cv2.fillPoly(mascara, [pts], 1)
        n += 1
    area, longitud, ancho = medir_mascara(mascara)
    return {"area_pct": 100 * area / (h * w), "longitud_px": longitud, "ancho_px": round(ancho, 2), "n_grietas": n}

In [ ]:
# PASO 9: Proceso base = grietas reales de las imágenes de ENTRENAMIENTO
tam = {Path(i["file"]).stem: (i["height"], i["width"]) for i in imagenes}

filas = []
for ruta in sorted((raiz / "labels" / "train").glob("*.txt")):
    h, w = tam[ruta.stem]
    filas.append(medir_etiqueta(ruta, h, w))
base = pd.DataFrame(filas)

print("Imágenes del proceso base:", len(base))
print("Imágenes sin ninguna grieta:", int((base.area_pct == 0).sum()))
print(base.describe().round(2))
base.area_pct.hist(bins=40)
plt.xlabel("% del área de la imagen con grieta"); plt.ylabel("Cantidad de imágenes"); plt.title("Distribución del daño en el proceso base")
plt.show()

In [ ]:
# PASO 10: Carta de control (SPC) y los 3 niveles de daño
mu = base.area_pct.mean()          # promedio del proceso (línea central)
sigma = base.area_pct.std()        # desviación estándar
LIM_ADV = mu + 2 * sigma           # límite de advertencia (2 sigma)
UCL = mu + 3 * sigma               # límite de control superior (3 sigma)

def nivel_dano(area_pct):
    if area_pct <= mu:
        return "BAJO"
    elif area_pct <= LIM_ADV:
        return "MEDIO"
    return "ALTO"

print(f"Promedio (mu): {mu:.2f}%   Sigma: {sigma:.2f}%")
print(f"BAJO  : área <= {mu:.2f}%")
print(f"MEDIO : {mu:.2f}% < área <= {LIM_ADV:.2f}%")
print(f"ALTO  : área > {LIM_ADV:.2f}%   (límite de control UCL = {UCL:.2f}%)")
print(base.area_pct.apply(nivel_dano).value_counts(normalize=True).round(3))

plt.figure(figsize=(12, 4))
plt.plot(base.area_pct.values[:150], marker="o", ms=3, lw=0.8, color="gray")
plt.axhline(mu, color="green", label="Promedio")
plt.axhline(LIM_ADV, color="orange", ls="--", label="Límite de advertencia (2σ)")
plt.axhline(UCL, color="red", ls="--", label="Límite de control UCL (3σ)")
plt.xlabel("Pieza inspeccionada"); plt.ylabel("% de área con grieta")
plt.title("Carta de control de individuos (primeras 150 piezas)")
plt.legend(); plt.show()

In [ ]:
# PASO 11: Probabilidad de falla y capacidad del proceso (Cpk)
from scipy import stats
USL = 5.0   # límite de especificación: una pieza FALLA si la grieta cubre más del 5% de la imagen

# el daño no es simétrico (casi todo es pequeño), por eso usamos la distribución lognormal
forma, _, escala = stats.lognorm.fit(base.area_pct[base.area_pct > 0], floc=0)

def prob_falla(factor=1.0):
    # probabilidad de que el área de grieta supere USL cuando el daño se multiplica por 'factor'
    return 1 - stats.lognorm.cdf(USL / factor, forma, loc=0, scale=escala)

cpk = (USL - mu) / (3 * sigma)
print(f"Probabilidad de falla del proceso (modelo) : {prob_falla():.1%}")
print(f"Piezas realmente fuera de especificación   : {(base.area_pct > USL).mean():.1%}")
print(f"Capacidad del proceso Cpk = {cpk:.2f} ->", "CAPAZ (>=1.33)" if cpk >= 1.33 else "NO CAPAZ (<1.33)")

In [ ]:
# PASO 12: Escenarios a futuro (sismo, suelo, construcción). OJO: son SUPUESTOS del modelo
escenarios = {
    "Normal (envejecimiento)": 1.15,
    "Construcción cercana": 1.5,
    "Movimiento de suelo": 2.0,
    "Sismo": 2.5,
}   # factor por el que se multiplica el área de la grieta en 12 meses

tabla = pd.DataFrame({
    "Factor de crecimiento": escenarios,
    "Prob. de falla": {k: prob_falla(f) for k, f in escenarios.items()},
    "% de piezas en nivel ALTO": {k: float((base.area_pct * f > LIM_ADV).mean()) for k, f in escenarios.items()},
})
print(tabla.round(3))

tabla["Prob. de falla"].plot(kind="bar", color=["green", "gold", "orange", "red"], rot=15)
plt.ylabel("Probabilidad de falla"); plt.title("Probabilidad de falla a 12 meses según el escenario")
plt.show()

## FASE IV: Validación y sensibilidad
Comparamos lo que mide el sistema (con YOLO) contra la grieta real etiquetada en las **111 imágenes de prueba**, que el modelo nunca vio.

In [ ]:
# PASO 13: Validación con las imágenes de PRUEBA (sistema vs. realidad)
filas = []
for r in sorted((raiz / "images" / "test").glob("*.jpg")):
    h, w = tam[r.stem]
    real = medir_etiqueta(raiz / "labels" / "test" / (r.stem + ".txt"), h, w)
    pred = medir_prediccion(mejor.predict(str(r), conf=0.25, verbose=False)[0])
    filas.append({"area_real": real["area_pct"], "area_pred": pred["area_pct"]})
val = pd.DataFrame(filas)
val["nivel_real"] = val.area_real.apply(nivel_dano)
val["nivel_pred"] = val.area_pred.apply(nivel_dano)

print("Correlación entre área real y área medida por YOLO:", round(val.area_real.corr(val.area_pred), 3))
print("Error promedio del área:", round((val.area_real - val.area_pred).abs().mean(), 2), "puntos de %")
print("Nivel de daño acertado (BAJO/MEDIO/ALTO):", round((val.nivel_real == val.nivel_pred).mean(), 3))
print("\nAcierto por nivel real:")
resumen = val.assign(acierto=(val.nivel_real == val.nivel_pred)).groupby("nivel_real")["acierto"].agg(["count", "mean"]).round(3)
resumen.columns = ["imágenes", "acierto"]
print(resumen)

plt.scatter(val.area_real, val.area_pred, alpha=0.6)
plt.plot([0, val.area_real.max()], [0, val.area_real.max()], "r--", label="Medición perfecta")
plt.xlabel("% de área real (etiqueta)"); plt.ylabel("% de área medida por YOLO")
plt.title("Validación en imágenes de prueba"); plt.legend(); plt.show()

In [ ]:
# PASO 14: Análisis de sensibilidad
import seaborn as sns

# (a) ¿Cómo cambia la probabilidad de falla (%) si cambia el límite aceptable (USL) y el crecimiento del daño?
usls = [3, 4, 5, 6, 8]
factores = [1.0, 1.15, 1.5, 2.0, 2.5]
sens = pd.DataFrame({f"x{f}": [100 * (1 - stats.lognorm.cdf(u / f, forma, loc=0, scale=escala)) for u in usls] for f in factores},
                    index=[f"USL={u}%" for u in usls])
sns.heatmap(sens, annot=True, fmt=".1f", cmap="Reds")
plt.title("Probabilidad de falla (%) según USL y factor de crecimiento"); plt.show()

# (b) ¿Cómo cambia la clasificación si movemos el límite de advertencia (k sigmas)?
for k in [1.5, 2, 2.5]:
    lim = mu + k * sigma
    niv = lambda a: "BAJO" if a <= mu else ("MEDIO" if a <= lim else "ALTO")
    acierto = (val.area_real.apply(niv) == val.area_pred.apply(niv)).mean()
    print(f"k={k}: límite={lim:.2f}% | piezas ALTO en el proceso base={(base.area_pct > lim).mean():.1%} | acierto del nivel en prueba={acierto:.1%}")

### Limitaciones del sistema (para la defensa)
- **Falsos positivos:** sombras, manchas, juntas o rayones pueden confundirse con grietas y aumentan el área medida.
- **Falsos negativos:** grietas muy finas o poco contrastadas no se detectan y el daño sale subestimado.
- **Iluminación y ángulo:** cambian lo que YOLO ve y mueven el % de área, y con ella el nivel de daño.
- **Sin escala real:** medimos en píxeles y en % de la imagen, no en milímetros. Una foto más cerca o más lejos cambia el resultado.
- **Dataset:** son calles y paredes, no cimientos.
- **Predicción a futuro:** el dataset no tiene tiempo ni sismos. Los factores de crecimiento (sismo, suelo, construcción) son **supuestos** que se pueden calibrar con inspecciones periódicas.
- **Incertidumbre:** el error de medición de YOLO entra en la carta de control y en la probabilidad de falla; por eso en la zona de frontera entre niveles el resultado es menos confiable.

In [ ]:
# PASO 15: DEMO - subir una imagen y recibir el diagnóstico de daño (BAJO / MEDIO / ALTO)
import gradio as gr

RECOMENDACION = {
    "BAJO": "APROBADO. Daño dentro de lo normal. Monitorear en la próxima inspección.",
    "MEDIO": "ALERTA. Sellar la grieta y volver a inspeccionar en 3 meses.",
    "ALTO": "RECHAZADO. Daño crítico. Requiere evaluación urgente de un ingeniero estructural.",
}

def diagnosticar(ruta):
    res = mejor.predict(ruta, conf=0.25, verbose=False)[0]
    med = medir_prediccion(res)
    area = med["area_pct"]
    nivel = nivel_dano(area)

    # carta de control con el punto de esta imagen (X negra)
    fig, ax = plt.subplots(figsize=(6, 3.2))
    ax.plot(base.area_pct.values[:100], marker="o", ms=2, lw=0.6, color="lightgray")
    ax.axhline(mu, color="green"); ax.axhline(LIM_ADV, color="orange", ls="--"); ax.axhline(UCL, color="red", ls="--")
    ax.scatter([100], [area], color="black", s=140, marker="X", zorder=5)
    ax.set_title("Carta de control (X negra = esta imagen)"); ax.set_ylabel("% de área con grieta")
    plt.close(fig)

    texto = (f"## Nivel de daño: {nivel}\n**{RECOMENDACION[nivel]}**\n\n"
             f"- Área con grieta: {area:.2f}% de la imagen\n- Longitud: {med['longitud_px']} px\n"
             f"- Ancho promedio: {med['ancho_px']} px\n- Grietas detectadas: {med['n_grietas']}\n\n"
             "### Proyección a 12 meses (supuestos)\n| Escenario | Área proyectada | Nivel | ¿Falla? | Prob. de falla del proceso |\n|---|---|---|---|---|\n")
    for nombre, f in escenarios.items():
        a2 = area * f
        texto += f"| {nombre} | {a2:.2f}% | {nivel_dano(a2)} | {'SÍ' if a2 > USL else 'No'} | {prob_falla(f):.1%} |\n"

    return np.ascontiguousarray(res.plot()[..., ::-1]), texto, fig

demo = gr.Interface(fn=diagnosticar,
                    inputs=gr.Image(type="filepath", label="Sube una foto de la grieta"),
                    outputs=[gr.Image(label="Grieta detectada por YOLO"), gr.Markdown(), gr.Plot(label="Carta de control")],
                    title="Detección de daño estructural por grietas")
demo.launch(debug=False)

In [ ]:
# PASO 16: Guardar el modelo entrenado (best.pt) para subirlo a GitHub
import shutil
shutil.copy(carpeta / "weights" / "best.pt", "best.pt")
print("best.pt guardado:", round(Path("best.pt").stat().st_size / 1e6, 1), "MB")

In [ ]:
# PASO 17: Probar el diagnóstico con una imagen de cada nivel (BAJO / MEDIO / ALTO)
from IPython.display import Markdown, display
import shutil

# buscar una imagen de cada nivel (primero en prueba, luego en entrenamiento)
elegidas = {}
candidatas = sorted((raiz / "images" / "test").glob("*.jpg")) + sorted((raiz / "images" / "train").glob("*.jpg"))
for r in candidatas:
    area = medir_prediccion(mejor.predict(str(r), conf=0.25, verbose=False)[0])["area_pct"]
    elegidas.setdefault(nivel_dano(area), r)
    if len(elegidas) == 3:
        break

Path("ejemplos_demo").mkdir(exist_ok=True)       # estas imágenes se pueden subir a la demo del PASO 15
for nivel, r in elegidas.items():
    shutil.copy(r, f"ejemplos_demo/{nivel}_{r.name}")
    imagen, texto, fig = diagnosticar(str(r))
    plt.figure(figsize=(5, 4)); plt.imshow(imagen); plt.axis("off"); plt.title(f"Nivel de daño: {nivel}"); plt.show()
    display(Markdown(texto))